In [ ]:
import xlwings as xw
import numpy as np
import pandas as pd
from scipy.stats import triang
import matplotlib.pyplot as plt
import os


In [ ]:
# import util functions from external util file
%run util.py

In [ ]:
def extract_data_from_folders(root_dir):
    data_dict = {}

    # Traverse the directory tree
    for folder_name in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder_name)
        
        # Check if it's a folder (skip if not a directory)
        if os.path.isdir(folder_path):
            # Extract the numeric ID from the folder name (assuming it starts with a number)
            folder_id = ''.join([ch for ch in folder_name if ch.isdigit()])  # Gets the digits from folder name
            
            # Initialize a list to store file details for this folder
            file_data = []
            
            # Iterate over files in the folder
            for file_name in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file_name)
                
                # Only process files (skip subfolders)
                if os.path.isfile(file_path):
                    # Extract the number after "MP" in the filename
                    if file_name.startswith('MP'):
                        file_number = ''.join([ch for ch in file_name.split('_')[0][2:] if ch.isdigit()])
                        file_size = os.path.getsize(file_path)  # Get the file size in bytes
                        
                        # Add the extracted data to the list
                        file_data.append({
                            'file_number': file_number,
                            'file_size': file_size
                        })
            
            # Store the extracted data in the dictionary
            data_dict[folder_id] = file_data
    
    return data_dict

# Define the root directory where the folders are located
root_directory = './raw_data'

# Call the function and get the dictionary
result_dict = extract_data_from_folders(root_directory)

print(result_dict)


In [ ]:

 # Path to the data file 
excel_file = './methodology_data.xlsx' 

 # Name of the sheet containing data
sheet_name = 'data' 

num_header_rows = 1 # number of rows with metadata or headers
num_header_columns = 3 # number of columns with metadata or headers


# Connect to Excel
# THIS OPENS AND EDITS THE EXCEL FILE ON YOUR COMPUTER!
wb = xw.Book(excel_file)
sheet = wb.sheets[sheet_name]

# Determine the range of columns and results
data_range = sheet.range("A1").expand()  # Automatically detect data range
num_columns = data_range.columns.count  # Total number of columns
num_rows = data_range.rows.count  # Total number of columns

In [ ]:
row_monitoring_period_raw_file_size = get_row_by_variable("monitoring_period_raw_file_size", sheet, num_rows) 
if (row_monitoring_period_raw_file_non_empty_cells == -1)
    except("Row monitoring_period_raw_file_non_empty_cells doesn't exist. Please add it manually.")
row_numeric_id = get_row_by_variable("numeric_id", sheet, num_rows) 
row_monitoring_period = get_row_by_variable("monitoring_period", sheet, num_rows) 
row_data_missing = get_row_by_variable("data_missing", sheet, num_rows)

for i in range(num_header_columns + 1, num_columns):
    if (int(sheet.cells(row_data_missing, i).value) == 1):
        continue
    
    project_id = str(int(sheet.cells(row_numeric_id, i).value))
    monitoring_period = str(int(sheet.cells(row_monitoring_period, i).value))
    try:
        sheet.cells(row_monitoring_period_raw_file_size, i).value = next(item['file_size'] for item in result_dict[project_id] if item['file_number'] == monitoring_period)
    except:
        print(project_id)
        print(monitoring_period)

In [ ]:

result_dict["8275"]


In [ ]:

row_monitoring_period_raw_file_size = get_row_by_variable("monitoring_period_raw_file_size", sheet, num_rows) 
row_year = get_row_by_variable("year", sheet, num_rows) 

file_size = sheet.range((row_monitoring_period_raw_file_size, num_header_columns + 1), (row_monitoring_period_raw_file_size, num_columns)).value
year = sheet.range((row_year, num_header_columns + 1), (row_year, num_columns)).value

In [ ]:
df = pd.DataFrame({
    'Year': year,
    'File Size': file_size
})

# Group by 'Year' and calculate the average file size for each year
average_file_sizes = df.groupby('Year')['File Size'].mean()

# Plot the result
plt.plot(average_file_sizes.index, average_file_sizes.values, marker='o', linestyle='-', color='b')  # Line plot with markers
plt.xlabel('Year')
plt.ylabel('Average File Size')
plt.title('Average File Size Over Years')
plt.grid(True)

# Show the plot
plt.show()